# v2000 final run — live/retrospective fleet monitor

Re-runnable any time (mid-run or after). Rebuilds everything from the run
directory alone: per-task progress pings in the logs + the cost-aware cells
manifest. Tracks the fleet's projected total-time distribution through time,
overall and by tail tier, plus current-status tables.

Set `RUN_START` to just before the launch you care about — older log files
(earlier attempts in the same dir) are excluded by modification time.

In [ ]:
from datetime import datetime
from pathlib import Path
import glob, json, os, re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from idd_tools.jobmon import inflate_cells

RUN_DIR   = Path('/mnt/team/idd/pub/idd_tc_mortality/02-evaluate/20260722_v2000_final')
MANIFEST  = RUN_DIR / 'cells_manifest_costaware.json'
RUN_START = datetime(2026, 7, 31, 14, 20)   # TUNE: just before this run's launch
GRID_MIN  = 3                                # time-series resolution, minutes

PING = re.compile(r"Cells progress: (\d+)/(\d+) \(\d+%\), elapsed (\d+)s")
print(f'{MANIFEST.name}, run start filter {RUN_START}')

In [ ]:
# Task -> tier composition (cost-packed tasks are mostly tier-pure)
doc = json.loads(MANIFEST.read_text())
rows = []
for i, task in enumerate(doc['tasks']):
    cs = inflate_cells(task['task_args'])
    tiers = pd.Series([c['tier'] for c in cs])
    rows.append({'task': i, 'n_cells': len(cs),
                 'tier': tiers.mode()[0],
                 'tier_purity': (tiers == tiers.mode()[0]).mean()})
task_df = pd.DataFrame(rows).set_index('task')
print(f"{len(task_df)} tasks; tier purity min {task_df.tier_purity.min():.0%}")
print(task_df.groupby('tier').agg(n_tasks=('n_cells','size'),
                                  cells=('n_cells','sum')).to_string())

In [ ]:
# Parse every ping from this run's logs into (task, wall_time, done, total)
recs = []
for f in glob.glob(str(RUN_DIR / 'logs' / 'task_*.err' / '*' / '*.e*')):
    mtime = os.path.getmtime(f)
    if datetime.fromtimestamp(mtime) < RUN_START:
        continue
    m_idx = re.search(r'task_index-(\d+)\.e', f)
    if not m_idx:
        continue
    task = int(m_idx.group(1))
    pings = [m for m in (PING.search(l) for l in open(f)) if m]
    if not pings:
        continue
    last_el = int(pings[-1].group(3))
    start = mtime - last_el                      # wall anchor (~1-2 min error)
    for m in pings:
        done, total, el = map(int, m.groups())
        recs.append({'task': task, 'wall': start + el, 'done': done,
                     'total': total, 'elapsed': el})
pings_df = pd.DataFrame(recs).sort_values(['task', 'wall'])
print(f'{len(pings_df):,} pings across {pings_df.task.nunique()} tasks')

In [ ]:
# Fleet time series: at each grid time, each task's projected total
def task_projection(g, t):
    g = g[g.wall <= t]
    if g.empty:
        return None
    last = g.iloc[-1]
    if last.done >= last.total:
        return {'est_total': last.elapsed / 60, 'done_frac': 1.0, 'rate': np.nan}
    prev = g.iloc[-2] if len(g) >= 2 else None
    if prev is not None and last.wall > prev.wall and last.done > prev.done:
        rate = (last.done - prev.done) / (last.wall - prev.wall)
    else:
        rate = last.done / last.elapsed if last.elapsed else np.nan
    eta = (last.total - last.done) / rate if rate and rate > 0 else np.nan
    return {'est_total': (last.elapsed + eta) / 60,
            'done_frac': last.done / last.total, 'rate': rate}

t0, t1 = pings_df.wall.min(), pings_df.wall.max()
grid = np.arange(t0 + 60, t1 + 60, GRID_MIN * 60)
ts_rows = []
for t in grid:
    per_task = {}
    for task, g in pings_df.groupby('task'):
        p = task_projection(g, t)
        if p:
            per_task[task] = p
    if not per_task:
        continue
    est = pd.Series({k: v['est_total'] for k, v in per_task.items()}).dropna()
    active = {k: v for k, v in per_task.items() if v['done_frac'] < 1.0}
    row = {'time': datetime.fromtimestamp(t), 'n_reporting': len(per_task),
           'n_finished': sum(v['done_frac'] >= 1.0 for v in per_task.values()),
           'est_min': est.min(), 'est_med': est.median(), 'est_max': est.max()}
    for tier, tasks in task_df.groupby('tier').groups.items():
        e = est[est.index.isin(tasks) & est.index.isin(active)]
        if len(e):
            row[f'med_{tier}'] = e.median()
    ts_rows.append(row)
ts = pd.DataFrame(ts_rows).set_index('time')
ts[['n_reporting', 'n_finished', 'est_min', 'est_med', 'est_max']].tail(8)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(ts.index, ts.est_min, ts.est_max, alpha=0.2, label='min-max')
ax.plot(ts.index, ts.est_med, lw=2, label='median')
ax.axhline(90, ls='--', c='crimson', label='90m ask')
ax.set_ylabel('projected task total (min)'); ax.set_yscale('log')
ax.set_title('Fleet projected total time, through time')
ax.legend(); fig.autofmt_xdate(); plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
for c in [c for c in ts.columns if c.startswith('med_')]:
    ax.plot(ts.index, ts[c], label=c.replace('med_', ''))
ax.axhline(90, ls='--', c='crimson')
ax.set_ylabel('median projected total (min), active tasks')
ax.set_yscale('log'); ax.set_title('By tail tier')
ax.legend(ncol=2, fontsize=8); fig.autofmt_xdate(); plt.show()

In [ ]:
# Current status: overall and by tier
latest = pings_df.sort_values('wall').groupby('task').last()
latest = latest.join(task_df[['tier']])
latest['finished'] = latest.done >= latest.total
status = (latest.groupby('tier')
    .agg(tasks=('done', 'size'), finished=('finished', 'sum'),
         cells_done=('done', 'sum'), cells_total=('total', 'sum'))
    .assign(pct=lambda d: (100 * d.cells_done / d.cells_total).round(1)))
print(status.to_string())
print(f"\nfleet: {int(latest.finished.sum())}/{len(task_df)} tasks finished, "
      f"{latest.done.sum():,}/{latest.total.sum():,} cells "
      f"({100 * latest.done.sum() / latest.total.sum():.1f}%)")